In [ ]:
# Step 1: Import libraries and set up environment

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

possible_dirs = [
    Path("."),
    Path("/home/course")
]

required_files = ["squirrel.csv", "hectare.csv", "stories.csv"]

data_dir = None

for folder in possible_dirs:
    if all((folder / file).exists() for file in required_files):
        data_dir = folder
        break

if data_dir is None:
    raise FileNotFoundError(
        "Cannot find squirrel.csv, hectare.csv and stories.csv. "
        "Please put them in the same folder as this Python file."
    )

print("Using data folder:", data_dir)

print("\n=== Step 1 Completed ===")

: 

In [ ]:
# Step 2: Load datasets and merge data

squirrel = pd.read_csv(data_dir / "squirrel.csv")
hectare = pd.read_csv(data_dir / "hectare.csv")
stories = pd.read_csv(data_dir / "stories.csv")

print("\n=== Dataset sizes ===")
print("squirrel shape:", squirrel.shape)
print("hectare shape:", hectare.shape)
print("stories shape:", stories.shape)

merge_keys = ["Hectare", "Shift", "Date"]

duplicated_keys = hectare.duplicated(subset=merge_keys).sum()
print("\nDuplicated merge keys in hectare:", duplicated_keys)

df = squirrel.merge(
    hectare,
    on=merge_keys,
    how="left",
    validate="many_to_one"
)

print("Merged data shape:", df.shape)

print("\n=== Step 2 Completed ===")

In [ ]:
# Step 3: Data cleaning and feature engineering
# This step prepares behavioural, spatial, temporal, and environmental features
# for later correlation analysis and supervised learning.

df_clean = df.copy()


# 3.1 Remove duplicate rows
before_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
after_rows = len(df_clean)

duplicates_removed = before_rows - after_rows

In [ ]:
# 3.2 Convert Date and create time features
df_clean["Date_raw"] = df_clean["Date"]

df_clean["Date"] = pd.to_datetime(
    df_clean["Date"].astype(str).str.zfill(8),
    format="%m%d%Y",
    errors="coerce"
)

df_clean["Weekday"] = df_clean["Date"].dt.day_name()
df_clean["Day"] = df_clean["Date"].dt.day
df_clean["Month"] = df_clean["Date"].dt.month

In [ ]:
# 3.3 Check missing values
missing_summary = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_percentage": df_clean.isna().mean() * 100
}).sort_values("missing_percentage", ascending=False)

In [ ]:
# 3.4 Drop columns with too many missing values
missing_threshold = 0.80

protected_cols = [
    "X", "Y", "Unique Squirrel ID", "Hectare", "Shift", "Date", "Date_raw",
    "Age", "Primary Fur Color", "Highlight Fur Color", "Location",
    "Above Ground Sighter Measurement",
    "Running", "Chasing", "Climbing", "Eating", "Foraging",
    "Kuks", "Quaas", "Moans", "Tail flags", "Tail twitches",
    "Approaches", "Indifferent", "Runs from",
    "Hectare Conditions", "Sighter Observed Weather Data",
    "Other Animal Sightings", "Litter",
    "Number of sighters", "Number of Squirrels", "Total Time of Sighting",
    "Weekday", "Day", "Month"
]

cols_to_drop = [
    col for col in df_clean.columns
    if df_clean[col].isna().mean() > missing_threshold
    and col not in protected_cols
]

df_clean = df_clean.drop(columns=cols_to_drop)

In [ ]:
# 3.5 Convert True/False columns into 0/1
behaviour_cols = [
    "Running", "Chasing", "Climbing", "Eating", "Foraging",
    "Kuks", "Quaas", "Moans",
    "Tail flags", "Tail twitches"
]

interaction_cols = [
    "Approaches", "Indifferent", "Runs from"
]

binary_cols = behaviour_cols + interaction_cols

for col in binary_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(False).astype(int)

In [ ]:
# 3.6 Clean above-ground height
if "Above Ground Sighter Measurement" in df_clean.columns:
    df_clean["Above_Ground_Height"] = (
        df_clean["Above Ground Sighter Measurement"]
        .replace(["FALSE", False], 0)
    )

    df_clean["Above_Ground_Height"] = pd.to_numeric(
        df_clean["Above_Ground_Height"],
        errors="coerce"
    )

    df_clean["Above_Ground_Height"] = df_clean["Above_Ground_Height"].fillna(0)

In [ ]:
# 3.7 Fill categorical missing values
categorical_cols = [
    "Age",
    "Primary Fur Color",
    "Highlight Fur Color",
    "Location",
    "Hectare Conditions",
    "Sighter Observed Weather Data",
    "Other Animal Sightings",
    "Litter",
    "Weekday"
]

for col in categorical_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("Unknown")

In [ ]:
# 3.8 Fill numeric missing values
numeric_cols = [
    "X",
    "Y",
    "Number of sighters",
    "Number of Squirrels",
    "Total Time of Sighting",
    "Hectare Squirrel Number",
    "Above_Ground_Height",
    "Day",
    "Month"
]

for col in numeric_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

In [ ]:
# 3.9 Create new features
df_clean["Activity_Score"] = df_clean[
    ["Running", "Chasing", "Climbing", "Eating", "Foraging"]
].sum(axis=1)

df_clean["Vocalisation_Score"] = df_clean[
    ["Kuks", "Quaas", "Moans"]
].sum(axis=1)

df_clean["Tail_Activity"] = df_clean[
    ["Tail flags", "Tail twitches"]
].sum(axis=1)

In [ ]:
# 3.10 Create spatial and density categories
density_q1 = df_clean["Number of Squirrels"].quantile(1/3)
density_q2 = df_clean["Number of Squirrels"].quantile(2/3)

df_clean["Squirrel_Density_Level"] = np.select(
    [
        df_clean["Number of Squirrels"] <= density_q1,
        df_clean["Number of Squirrels"] <= density_q2,
        df_clean["Number of Squirrels"] > density_q2
    ],
    [
        "Low",
        "Medium",
        "High"
    ],
    default="Unknown"
)

y_q1 = df_clean["Y"].quantile(1/3)
y_q2 = df_clean["Y"].quantile(2/3)

df_clean["Park_NorthSouth_Zone"] = np.select(
    [
        df_clean["Y"] <= y_q1,
        df_clean["Y"] <= y_q2,
        df_clean["Y"] > y_q2
    ],
    [
        "South",
        "Middle",
        "North"
    ],
    default="Unknown"
)

x_median = df_clean["X"].median()

df_clean["Park_EastWest_Zone"] = np.where(
    df_clean["X"] <= x_median,
    "West",
    "East"
)

In [ ]:
# 3.11 Remove invalid coordinates
valid_coord_mask = (
    df_clean["X"].between(-74.1, -73.8) &
    df_clean["Y"].between(40.6, 40.9)
)

invalid_coords = (~valid_coord_mask).sum()
df_clean = df_clean[valid_coord_mask].copy()

In [ ]:
# 3.12 Final checks
important_check_cols = [
    "Age",
    "Primary Fur Color",
    "Location",
    "X",
    "Y",
    "Number of Squirrels",
    "Total Time of Sighting",
    "Activity_Score",
    "Vocalisation_Score",
    "Tail_Activity"
]

important_check_cols = [
    col for col in important_check_cols
    if col in df_clean.columns
]

# 3.13 Save cleaned dataset
df_clean.to_csv(data_dir / "A2_cleaned_full_dataset.csv", index=False)

In [ ]:
# Final summary
print("\n=== Preprocessing summary ===")
print("Original merged rows:", len(df))
print("Cleaned rows:", len(df_clean))
print("Columns after cleaning:", df_clean.shape[1])
print("Duplicate rows removed:", duplicates_removed)
print("Invalid coordinate rows removed:", invalid_coords)

print("\nDropped high-missing columns:")
print(cols_to_drop)

print("\nInteraction counts:")
print(df_clean[["Approaches", "Indifferent", "Runs from"]].sum())

print("\nMissing values in key columns:")
print(df_clean[important_check_cols].isna().sum())

print("\nFile saved:")
print("A2_cleaned_full_dataset.csv")

print("\n=== Step 3 Completed ===")

In [ ]:
# Step 4: Create target variable

model_df = df_clean.copy()

# Create conditions for each interaction behaviour
# 1 means the behaviour was observed
conditions = [
    model_df['Approaches'] == 1,
    model_df['Runs from'] == 1,
    model_df['Indifferent'] == 1
]

# Assign class labels for each condition
choices = [
    'Approach',
    'Run_From',
    'Ignore'
]

# Combine the three binary interaction columns
# into one multiclass target variable
model_df['Interaction_Type'] = np.select(
    conditions,
    choices,
    default='Unknown'
)

# Remove rows without a valid interaction label
model_df = model_df[
    model_df['Interaction_Type'] != 'Unknown'
].copy()

# Check class distribution
print(model_df['Interaction_Type'].value_counts())

print("\n=== Step 4 Completed ===")

In [ ]:
# Step 5: This step explores the relationships 
# between squirrel-human interaction behaviour and behavioural, spatial, and temporal features

plt.figure(figsize=(8, 5))

# 5.1 Plot the number of samples

sns.countplot(
    data=model_df,
    x='Interaction_Type',
    hue='Interaction_Type',
    palette='Set2',
    legend=False
)

plt.title('Distribution of Squirrel-Human Interaction Types')
plt.xlabel('Interaction Type')
plt.ylabel('Count')

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()
plt.savefig(data_dir / "step5_interaction_distribution.png", dpi=300)

In [ ]:
# 5.2 Correlation analysis of behavioural features

# Select behavioural features for correlation analysis
behaviour_features = [
    'Running',
    'Chasing',
    'Climbing',
    'Eating',
    'Foraging',
    'Kuks',
    'Quaas',
    'Moans',
    'Tail flags',
    'Tail twitches',
    'Activity_Score',
    'Vocalisation_Score',
    'Tail_Activity'
]

# Compute correlation matrix
correlation_matrix = model_df[
    behaviour_features
].corr()

plt.figure(figsize=(12, 8))

# Plot correlation heatmap
sns.heatmap(
    correlation_matrix,
    cmap='coolwarm',
    annot=True,
    fmt='.2f'
)

plt.title('Correlation Heatmap of Behavioural Features')
plt.tight_layout()

plt.show()
plt.savefig(data_dir / "step5_behaviour_correlation_heatmap.png", dpi=300)

In [ ]:
# 5.3 Compare activity score across interaction types

plt.figure(figsize=(8, 5))

# Plot boxplot of activity score for each interaction type
sns.boxplot(
    data=model_df,
    x='Interaction_Type',
    y='Activity_Score',
    palette='Set3'
)

plt.title('Activity Score by Interaction Type')
plt.xlabel('Interaction Type')
plt.ylabel('Activity Score')

plt.tight_layout()
plt.savefig(data_dir / "step5_activity_score_boxplot.png", dpi=300)

In [ ]:
# 5.4 Spatial distribution of interaction behaviour

plt.figure(figsize=(8, 8))

# Plot squirrel locations coloured by interaction type
sns.scatterplot(
    data=model_df,
    x='X',
    y='Y',
    hue='Interaction_Type',
    alpha=0.7
)

plt.title('Spatial Distribution of Interaction Behaviours')
plt.xlabel('Longitude')
plt.ylabel('Latitude')

plt.tight_layout()

plt.show()
plt.savefig(data_dir / "step5_spatial_distribution.png", dpi=300)

In [ ]:
# 5.5 Chi-square analysis

# Create contingency table between fur colour and interaction type
contingency_table = pd.crosstab(
    model_df['Primary Fur Color'],
    model_df['Interaction_Type']
)

# Perform chi-square test
chi2, p, dof, expected = chi2_contingency(contingency_table)

print('Chi-square statistic:', chi2)
print('p-value:', p)

print("\n=== Step 5 Completed ===")

In [ ]:
# Step 6: Prepare the dataset for supervised learning

# 6.1 Select features and target
# Select behavioural, spatial, and temporal features
selected_features = [
    'Running',
    'Chasing',
    'Climbing',
    'Eating',
    'Foraging',
    'Kuks',
    'Quaas',
    'Moans',
    'Tail flags',
    'Tail twitches',
    'Activity_Score',
    'Vocalisation_Score',
    'Tail_Activity',
    'X',
    'Y',
    'Shift',
    'Weekday',
    'Month',
    'Park_NorthSouth_Zone',
    'Park_EastWest_Zone',
    'Squirrel_Density_Level',
    'Primary Fur Color',
    'Location'
]

# Create feature matrix
X = model_df[selected_features]

# Create target variable
y = model_df['Interaction_Type']

In [ ]:
# 6.2 Split training and testing data

# Split dataset into training and testing sets
# stratify=y keeps class distribution balanced
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('Training set shape:', X_train.shape)
print('Testing set shape:', X_test.shape)

In [ ]:
# 6.3 Encode categorical features

# Define categorical columns
categorical_features = [
    'Shift',
    'Weekday',
    'Park_NorthSouth_Zone',
    'Park_EastWest_Zone',
    'Squirrel_Density_Level',
    'Primary Fur Color',
    'Location'
]

# Define numeric columns
numeric_features = [
    col for col in selected_features
    if col not in categorical_features
]

# Apply One-Hot Encoding to categorical features
preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        )
    ],
    remainder='passthrough'
)

# Fit encoder on training data
X_train_encoded = preprocessor.fit_transform(X_train)

# Transform testing data using same encoder
X_test_encoded = preprocessor.transform(X_test)

In [ ]:
# 6.4 Feature scaling

# Standardise feature values
# with_mean=False is required for sparse matrices
scaler = StandardScaler(with_mean=False)

# Fit scaler on training data
X_train_scaled = scaler.fit_transform(X_train_encoded)

# Apply same scaling to testing data
X_test_scaled = scaler.transform(X_test_encoded)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

In [ ]:
# 6.5 Cross validation with KNN

# Create KNN classification model
# n_neighbors=5 means the model checks 5 nearest neighbours
knn = KNeighborsClassifier(n_neighbors=5)

# Perform 5-fold cross validation
scores = cross_val_score(
    knn,
    X_train_scaled,
    y_train,
    cv=5,
    scoring='f1_weighted'
)

# Print F1-score from each fold
print('Cross-validation scores:', scores)

# Print average F1-score
print('Mean CV score:', scores.mean())

print("\n=== Step 6 Completed ===")

In [ ]:
# Step 7: Build and train classification models

# =========================
# Model 1A: Decision Tree
# =========================

decision_tree = DecisionTreeClassifier(

    max_depth=5,
    random_state=42,
    class_weight="balanced"
)

# Decision Tree does not require scaling
decision_tree.fit(X_train_encoded, y_train)

dt_train_pred = decision_tree.predict(X_train_encoded)
dt_test_pred = decision_tree.predict(X_test_encoded)

In [ ]:
# =========================
# Model 1B: Random Forest
# =========================

random_forest = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

# Random Forest does not require scaling
random_forest.fit(X_train_encoded, y_train)

rf_train_pred = random_forest.predict(X_train_encoded)
rf_test_pred = random_forest.predict(X_test_encoded)

In [ ]:
# Model 2: Logistic Regression

logistic_regression = LogisticRegression(
    max_iter=5000,
    random_state=42,
    class_weight="balanced"
)

# Logistic Regression requires scaled data
logistic_regression.fit(X_train_scaled, y_train)

lr_train_pred = logistic_regression.predict(X_train_scaled)
lr_test_pred = logistic_regression.predict(X_test_scaled)

In [ ]:
# Model 3: KNN

knn = KNeighborsClassifier(
    n_neighbors=5
)

# KNN requires scaled data
knn.fit(X_train_scaled, y_train)

knn_train_pred = knn.predict(X_train_scaled)
knn_test_pred = knn.predict(X_test_scaled)

In [ ]:
# Summary

print("\n=== Step 7 summary ===")
print("Decision Tree trained.")
print("Random Forest trained.")
print("Logistic Regression trained.")
print("KNN trained.")

print("\nPrediction results created:")
print("dt_train_pred, dt_test_pred")
print("rf_train_pred, rf_test_pred")
print("lr_train_pred, lr_test_pred")
print("knn_train_pred, knn_test_pred")

print("\n=== Step 7 Completed ===")

In [ ]:
# Step 8: Model Performance Evaluation

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

In [ ]:
# 8.1 Create evaluation function

def evaluate_model(y_true, y_pred, model_name):

    accuracy = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average='weighted'
    )

    recall = recall_score(
        y_true,
        y_pred,
        average='weighted'
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average='weighted'
    )

    print(f"\n=== {model_name} ===")
    print("Accuracy:", round(accuracy, 4))
    print("Precision:", round(precision, 4))
    print("Recall:", round(recall, 4))
    print("F1-score:", round(f1, 4))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    }

In [ ]:
# 8.2 Evaluate all models

dt_results = evaluate_model(
    y_test,
    dt_test_pred,
    "Decision Tree"
)

rf_results = evaluate_model(
    y_test,
    rf_test_pred,
    "Random Forest"
)

lr_results = evaluate_model(
    y_test,
    lr_test_pred,
    "Logistic Regression"
)

knn_results = evaluate_model(
    y_test,
    knn_test_pred,
    "KNN"
)

In [ ]:
# 8.3 Create confusion matrices

models = {
    "Decision Tree": dt_test_pred,
    "Random Forest": rf_test_pred,
    "Logistic Regression": lr_test_pred,
    "KNN": knn_test_pred
}

for model_name, predictions in models.items():

    cm = confusion_matrix(y_test, predictions)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=np.unique(y_test)
    )

    disp.plot()

    plt.title(f"{model_name} Confusion Matrix")

    plt.tight_layout()

    filename = (
        model_name.lower()
        .replace(" ", "_")
        + "_confusion_matrix.png"
    )
    
    plt.savefig(data_dir / filename, dpi=300)

    plt.close()

In [ ]:
# 8.4 Create performance comparison table

results_df = pd.DataFrame([
    dt_results,
    rf_results,
    lr_results,
    knn_results
])

print("\n=== Model Comparison Table ===")
print(results_df)

In [ ]:
# 8.5 Plot model performance comparison

results_melted = results_df.melt(
    id_vars='Model',
    var_name='Metric',
    value_name='Score'
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=results_melted,
    x='Metric',
    y='Score',
    hue='Model'
)

plt.ylim(0, 1)

plt.title('Model Performance Comparison')

plt.tight_layout()

plt.show()
plt.savefig(
    data_dir / "step8_model_performance_comparison.png",
    dpi=300
)

plt.close()

In [ ]:
# 8.6 Cross Validation

from sklearn.model_selection import cross_val_score

print("\n=== Cross Validation Results ===")

# Decision Tree CV
dt_cv_scores = cross_val_score(
    decision_tree,
    X_train_encoded,
    y_train,
    cv=5,
    scoring='f1_weighted'
)

print("\nDecision Tree CV Scores:")
print(dt_cv_scores)
print("Mean F1:", round(dt_cv_scores.mean(), 4))


# Random Forest CV
rf_cv_scores = cross_val_score(
    random_forest,
    X_train_encoded,
    y_train,
    cv=5,
    scoring='f1_weighted'
)

print("\nRandom Forest CV Scores:")
print(rf_cv_scores)
print("Mean F1:", round(rf_cv_scores.mean(), 4))


# Logistic Regression CV
lr_cv_scores = cross_val_score(
    logistic_regression,
    X_train_scaled,
    y_train,
    cv=5,
    scoring='f1_weighted'
)

print("\nLogistic Regression CV Scores:")
print(lr_cv_scores)
print("Mean F1:", round(lr_cv_scores.mean(), 4))


# KNN CV
knn_cv_scores = cross_val_score(
    knn,
    X_train_scaled,
    y_train,
    cv=5,
    scoring='f1_weighted'
)

print("\nKNN CV Scores:")
print(knn_cv_scores)
print("Mean F1:", round(knn_cv_scores.mean(), 4))


print("\n=== Step 8 Completed ===")

In [ ]:
# Step 9: Feature Importance Analysis and Feature Selection

# 9.1 Get feature names after encoding

feature_names = preprocessor.get_feature_names_out()

print("\nNumber of encoded features:", len(feature_names))

In [ ]:
# 9.2 Random Forest feature importance

rf_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": random_forest.feature_importances_
})

rf_importance_df = rf_importance_df.sort_values(
    by="Importance",
    ascending=False
)

print("\n=== Top 20 Important Features from Random Forest ===")
print(rf_importance_df.head(20))


# Save feature importance table
rf_importance_df.to_csv(
    data_dir / "step9_random_forest_feature_importance.csv",
    index=False
)

In [ ]:
# 9.3 Plot Top N important features

top_n = 15

plt.figure(figsize=(10, 6))

sns.barplot(
    data=rf_importance_df.head(top_n),
    x="Importance",
    y="Feature"
)

plt.title(f"Top {top_n} Feature Importances from Random Forest")
plt.xlabel("Feature Importance")
plt.ylabel("Feature")

plt.tight_layout()

plt.show()
plt.savefig(
    data_dir / "step9_top_feature_importance.png",
    dpi=300
)

plt.close()

In [ ]:
# 9.4 Select Top N features

selected_top_features = rf_importance_df.head(top_n)["Feature"].tolist()

print(f"\nSelected Top {top_n} features:")
print(selected_top_features)

In [ ]:
# 9.5 Create reduced datasets

# X_train_encoded and X_test_encoded may be sparse matrices.
# Convert them into DataFrames using encoded feature names.

X_train_encoded_df = pd.DataFrame(
    X_train_encoded.toarray()
    if hasattr(X_train_encoded, "toarray")
    else X_train_encoded,
    columns=feature_names
)

X_test_encoded_df = pd.DataFrame(
    X_test_encoded.toarray()
    if hasattr(X_test_encoded, "toarray")
    else X_test_encoded,
    columns=feature_names
)

X_train_top = X_train_encoded_df[selected_top_features]
X_test_top = X_test_encoded_df[selected_top_features]

In [ ]:
# 9.6 Retrain model using selected features

rf_top = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    class_weight="balanced"
)

rf_top.fit(X_train_top, y_train)

rf_top_pred = rf_top.predict(X_test_top)

In [ ]:
# 9.7 Evaluate reduced-feature model

rf_top_results = evaluate_model(
    y_test,
    rf_top_pred,
    f"Random Forest Top {top_n} Features"
)

In [ ]:
# 9.8 Compare full model vs selected-feature model

feature_selection_results_df = pd.DataFrame([
    rf_results,
    rf_top_results
])

print("\n=== Full Features vs Selected Features ===")
print(feature_selection_results_df)


feature_selection_melted = feature_selection_results_df.melt(
    id_vars="Model",
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(9, 5))

sns.barplot(
    data=feature_selection_melted,
    x="Metric",
    y="Score",
    hue="Model"
)

plt.ylim(0, 1)

plt.title("Random Forest Performance: Full Features vs Top Features")

plt.tight_layout()

plt.show()
plt.savefig(
    data_dir / "step9_feature_selection_comparison.png",
    dpi=300
)

plt.close()

In [ ]:
# 9.9 Confusion matrix for selected-feature model

cm = confusion_matrix(y_test, rf_top_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=random_forest.classes_
)

disp.plot()

plt.title(f"Random Forest Top {top_n} Features Confusion Matrix")

plt.tight_layout()

plt.show()
plt.savefig(
    data_dir / "step9_rf_top_features_confusion_matrix.png",
    dpi=300
)

plt.close()


print("\n=== Step 9 Completed ===")
print("Feature importance plot saved.")
print("Feature selection comparison plot saved.")